<a href="https://colab.research.google.com/github/suhanigupta786/Flight-Club/blob/main/06_XGBoost_Flight_Delay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TASK 6 — XGBoost Flight Delay Classification

**Input:** the same `task4_outputs.zip` used for Task 5 Logistic Regression. **Splits:** January–August train (217,727), September–October validation (55,628), November–December test (53,991). **Goal:** compare XGBoost fairly with Logistic Regression using the exact same preprocessed features and labels.

This notebook trains on **training only**, chooses early stopping and an F1 decision threshold using **validation only**, then evaluates the held-out test. Do not optimize against test results.

## 1. Setup and extract Task 4 artifacts
Upload `task4_outputs.zip` to the Colab **Files** panel first. This notebook does not require the huge Task 4 dense CSV.

In [4]:
%pip -q install "xgboost>=2.0,<4"

In [5]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import joblib
from scipy import sparse
import xgboost as xgb
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, precision_recall_curve)

ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
OUT = ROOT / 'task4_outputs'
ZIP = ROOT / 'task4_outputs.zip'
if not OUT.is_dir():
    assert ZIP.exists(), f'Upload {ZIP.name} in the Colab Files panel first.'
    with zipfile.ZipFile(ZIP) as archive:
        # Unpack only expected local artifacts, under the working directory.
        for member in archive.infolist():
            target = (ROOT / member.filename).resolve()
            assert target == ROOT.resolve() or ROOT.resolve() in target.parents, 'Unsafe archive path'
        archive.extractall(ROOT)
assert OUT.is_dir(), f'Could not find {OUT}. ZIP must contain task4_outputs/.'
print('XGBoost version:', xgb.__version__)
print('Artifacts:', sorted(p.name for p in OUT.iterdir()))

XGBoost version: 3.4.1
Artifacts: ['X_test_cat_dl.npy', 'X_test_num_dl.npy', 'X_test_processed.npz', 'X_train_cat_dl.npy', 'X_train_num_dl.npy', 'X_train_processed.npz', 'X_val_cat_dl.npy', 'X_val_num_dl.npy', 'X_val_processed.npz', 'dl_preprocessing.joblib', 'preprocessor.joblib', 'processed_feature_names.csv', 'test_metadata.csv', 'train_metadata.csv', 'val_metadata.csv', 'y_test.npy', 'y_train.npy', 'y_val.npy']


## 2. Load processed matrices and validate chronological splits

In [6]:
def load_processed(split):
    npz = OUT / f'X_{split}_processed.npz'
    npy = OUT / f'X_{split}_processed.npy'
    if npz.exists():
        return sparse.load_npz(npz).tocsr()
    if npy.exists():
        return np.load(npy, mmap_mode='r')
    raise FileNotFoundError(f'No processed feature matrix for {split} in {OUT}')

X_train, X_val, X_test = (load_processed(s) for s in ('train', 'val', 'test'))
y_train, y_val, y_test = (np.load(OUT / f'y_{s}.npy').astype(int) for s in ('train', 'val', 'test'))
feature_names = pd.read_csv(OUT / 'processed_feature_names.csv')['feature'].tolist()
preprocessor = joblib.load(OUT / 'preprocessor.joblib')

splits = {'train': (X_train, y_train, 217727, set(range(1, 9))),
          'val': (X_val, y_val, 55628, {9, 10}),
          'test': (X_test, y_test, 53991, {11, 12})}
for s, (X, y, expected_n, months) in splits.items():
    metadata = pd.read_csv(OUT / f'{s}_metadata.csv', parse_dates=['date'])
    assert X.shape[0] == len(y) == len(metadata) == expected_n, f'{s} rows mismatch'
    assert X.shape[1] == len(feature_names), f'{s} features mismatch'
    assert set(metadata['split'].unique()) == {s}
    assert set(metadata['date'].dt.month.unique()) == months
    assert np.array_equal(metadata['delayed'].to_numpy(dtype=int), y), f'{s} target order mismatch'
    assert set(np.unique(y)).issubset({0, 1})
    vals = X.data if sparse.issparse(X) else X
    assert np.isfinite(vals).all(), f'{s} contains NaN or infinity'
    print(f'{s}: {X.shape} | delay rate {y.mean():.2%}')
assert len(set(feature_names)) == len(feature_names)
for forbidden in ('arr_delay', 'dep_delay', 'arr_time', 'dep_time', 'air_time', 'row_id', 'delayed', 'status', 'cancelled'):
    assert all(name != forbidden and not name.endswith('__' + forbidden) for name in feature_names), forbidden
print('Chronological split and basic leakage checks passed')

train: (217727, 356) | delay rate 25.61%
val: (55628, 356) | delay rate 15.05%
test: (53991, 356) | delay rate 24.99%
Chronological split and basic leakage checks passed


## 3. Fit XGBoost (validation-based early stopping)
The model uses `tree_method="hist"` for faster training. The training class ratio informs `scale_pos_weight`. The validation split is used for early stopping, not training.

In [7]:
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos
model = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=5,
    reg_lambda=2.0,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='hist',
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=40,
    random_state=42,
    n_jobs=4,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print('Best boosting iteration:', model.best_iteration)
print('Validation best score:', model.best_score)
print('Training positive rate:', round(y_train.mean(), 4))

Best boosting iteration: 82
Validation best score: 0.2657513827484439
Training positive rate: 0.2561


## 4. Baseline metrics at threshold 0.50

In [11]:
def metric_row(name, y, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {'split': name, 'threshold': float(threshold), 'n': int(len(y)),
            'positive_rate': float(y.mean()),
            'accuracy': float(accuracy_score(y, pred)),
            'precision': float(precision_score(y, pred, zero_division=0)),
            'recall': float(recall_score(y, pred, zero_division=0)),
            'f1': float(f1_score(y, pred, zero_division=0)),
            'roc_auc': float(roc_auc_score(y, proba)),
            'pr_auc': float(average_precision_score(y, proba))}

val_prob = model.predict_proba(X_val)[:, 1]
val_baseline = metric_row('val', y_val, val_prob, 0.5)
print('Validation baseline at 0.5:')
display(pd.DataFrame([val_baseline]))

Validation baseline at 0.5:


,split,threshold,n,positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc
0,val,0.5,55628,0.150518,0.700582,0.243226,0.46853,0.320219,0.661729,0.265836


## 5. Tune F1 threshold on validation only
Keep the Logistic Regression baseline threshold (0.5) for comparison. This search uses only validation labels and probabilities.

In [12]:
precision, recall, candidate_thresholds = precision_recall_curve(y_val, val_prob)
assert len(candidate_thresholds) > 0
scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_idx = int(np.argmax(scores))
best_threshold = float(candidate_thresholds[best_idx])
val_tuned = metric_row('val_tuned', y_val, val_prob, best_threshold)
print(f'Validation-selected F1 threshold: {best_threshold:.6f}')
display(pd.DataFrame([val_baseline, val_tuned]))

Validation-selected F1 threshold: 0.465672


,split,threshold,n,positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc
0,val,0.500000,55628,0.150518,0.700582,0.243226,0.468530,0.320219,0.661729,0.265836
1,val_tuned,0.465672,55628,0.150518,0.658787,0.231769,0.547355,0.325647,0.661729,0.265836


## 6. Final held-out test evaluation
Run this only after the model and threshold choices are finalized. ROC-AUC and PR-AUC are unchanged by the classification threshold.

In [13]:
test_prob = model.predict_proba(X_test)[:, 1]
test_baseline = metric_row('test', y_test, test_prob, 0.5)
test_tuned = metric_row('test_tuned', y_test, test_prob, best_threshold)
results = pd.DataFrame([val_baseline, val_tuned, test_baseline, test_tuned])
display(results)
for label, threshold in [('TEST BASELINE', 0.5), ('TEST TUNED', best_threshold)]:
    pred = (test_prob >= threshold).astype(int)
    print(f'\n{label} (threshold={threshold:.4f})')
    print('Confusion matrix [TN FP; FN TP]:\n', confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, zero_division=0))

,split,threshold,n,positive_rate,accuracy,precision,recall,f1,roc_auc,pr_auc
0,val,0.500000,55628,0.150518,0.700582,0.243226,0.468530,0.320219,0.661729,0.265836
1,val_tuned,0.465672,55628,0.150518,0.658787,0.231769,0.547355,0.325647,0.661729,0.265836
2,test,0.500000,53991,0.249894,0.661629,0.345774,0.396902,0.369578,0.628759,0.342953
3,test_tuned,0.465672,53991,0.249894,0.630012,0.339187,0.506819,0.406395,0.628759,0.342953



TEST BASELINE (threshold=0.5000)
Confusion matrix [TN FP; FN TP]:
 [[30367 10132]
 [ 8137  5355]]
              precision    recall  f1-score   support

           0       0.79      0.75      0.77     40499
           1       0.35      0.40      0.37     13492

    accuracy                           0.66     53991
   macro avg       0.57      0.57      0.57     53991
weighted avg       0.68      0.66      0.67     53991


TEST TUNED (threshold=0.4657)
Confusion matrix [TN FP; FN TP]:
 [[27177 13322]
 [ 6654  6838]]
              precision    recall  f1-score   support

           0       0.80      0.67      0.73     40499
           1       0.34      0.51      0.41     13492

    accuracy                           0.63     53991
   macro avg       0.57      0.59      0.57     53991
weighted avg       0.69      0.63      0.65     53991



## 7. Compare directly with Task 5 Logistic Regression
The reference metrics below are from your previous run (test baseline and validation-selected tuned threshold). They are kept as comparison-only values; Task 6 does not retrain Logistic Regression.

In [14]:
lr_reference = pd.DataFrame([
    {'model': 'Logistic Regression', 'variant': 'baseline', 'f1': 0.392013,
     'recall': 0.473614, 'precision': 0.334397, 'roc_auc': 0.618679, 'pr_auc': 0.338730},
    {'model': 'Logistic Regression', 'variant': 'tuned', 'f1': 0.405688,
     'recall': 0.540246, 'precision': 0.324793, 'roc_auc': 0.618679, 'pr_auc': 0.338730},
])
xgb_ref = results[results['split'].isin(['test', 'test_tuned'])].copy()
xgb_ref['variant'] = xgb_ref['split'].map({'test': 'baseline', 'test_tuned': 'tuned'})
xgb_ref['model'] = 'XGBoost'
comparison = pd.concat([lr_reference, xgb_ref], ignore_index=True)
display(comparison[['model', 'variant', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']])
print('Note: choose your preferred model based on the application trade-off, not test-set tuning.')

,model,variant,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,baseline,0.334397,0.473614,0.392013,0.618679,0.338730
1,Logistic Regression,tuned,0.324793,0.540246,0.405688,0.618679,0.338730
2,XGBoost,baseline,0.345774,0.396902,0.369578,0.628759,0.342953
3,XGBoost,tuned,0.339187,0.506819,0.406395,0.628759,0.342953


Note: choose your preferred model based on the application trade-off, not test-set tuning.


## 8. Save reproducible model, metrics, and feature importances
The bundle includes the Task 4 fitted preprocessor for inference on the **original 21 Task 3 features** and the validation-selected threshold.

In [15]:
MODEL_OUT = ROOT / 'task6_outputs'
MODEL_OUT.mkdir(parents=True, exist_ok=True)
bundle = {
    'model': model, 'preprocessor': preprocessor,
    'feature_names': feature_names,
    'threshold': float(best_threshold),
    'best_iteration': int(model.best_iteration),
    'input_note': 'preprocessor.transform expects original 21 Task 3 model features',
}
joblib.dump(bundle, MODEL_OUT / 'xgboost_bundle.joblib')
results.to_csv(MODEL_OUT / 'xgboost_metrics.csv', index=False)
comparison.to_csv(MODEL_OUT / 'model_comparison.csv', index=False)
pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_}) \
    .sort_values('importance', ascending=False).to_csv(MODEL_OUT / 'xgboost_feature_importance.csv', index=False)
print('Saved:', [p.name for p in sorted(MODEL_OUT.iterdir())])

Saved: ['model_comparison.csv', 'xgboost_bundle.joblib', 'xgboost_feature_importance.csv', 'xgboost_metrics.csv']


## 9. Download outputs (small)
Only download the compact bundle, metrics and feature importances. Do **not** export a huge dense processed CSV.

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT / 'Task6_XGBoost'), 'zip',
                              root_dir=str(ROOT), base_dir='task6_outputs')
print('ZIP:', archive, 'Size MB:', round(Path(archive).stat().st_size / 1024**2, 2))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Not running inside Colab; ZIP is saved locally.')

## Report notes
- Training: Jan–Aug 2013; validation: Sep–Oct; final test: Nov–Dec.
- Imputation, scaling, and one-hot encoding come from Task 4 and are **not refitted**.
- Early stopping and F1 threshold selection use **validation only**.
- Compare XGBoost with Task 5 using the same split, label definition, and evaluation metrics.
- A two-day historical lag is a conservative approximation; document its prediction-time availability assumption.
- Actual performance values appear after executing the notebook with Task 4 artifacts.